# TurboQuant Visualized
**Paper**: [TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate](https://arxiv.org/abs/2504.19874)

1. The rotation trick (3D + high-d, general vectors)
2. The full compress → decompress pipeline (multiple examples)
3. Scalar quantization — what centroids mean
4. The bias problem
5. The two-stage fix (bias vs variance)
6. Near-optimality
7. KV cache: the full GPU memory picture

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.cluster.vq import kmeans
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 5)

## 1. The Rotation Trick
Randomly rotate any vector on the unit sphere → each coordinate becomes ~Gaussian and independent.

Shown first in 3D (visible), then high-d (where it matters). Using general vectors, not worst-case.

In [ ]:
# --- Math: 3D + high-d rotation ---
# 3D example
x_3d = np.array([0.8, 0.5, 0.3])
x_3d = x_3d / np.linalg.norm(x_3d)
Pi_3d, _ = np.linalg.qr(np.random.randn(3, 3))
y_3d = Pi_3d @ x_3d

# High-d example: general random vector (NOT worst-case)
d = 500
x = np.random.randn(d); x = x / np.linalg.norm(x)
Pi, _ = np.linalg.qr(np.random.randn(d, d))
y = Pi @ x
x_recovered = Pi.T @ y  # inverse rotation is just transpose

print(f'3D: original={x_3d.round(3)}, rotated={y_3d.round(3)}')
print(f'3D: ||x||={np.linalg.norm(x_3d):.3f}, ||Πx||={np.linalg.norm(y_3d):.3f} (preserved)')
print(f'\nHigh-d: rotated std={y.std():.4f} (expected {1/np.sqrt(d):.4f})')
print(f'Recovery error ||x - Πᵀy||: {np.linalg.norm(x - x_recovered):.1e} (lossless)')

In [ ]:
# --- Plot: 3D sphere + high-d coords ---
fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(131, projection='3d')
u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:20j]
ax1.plot_wireframe(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v), alpha=0.07, color='gray')
ax1.quiver(0,0,0, *x_3d, color='#e74c3c', lw=3, arrow_length_ratio=0.1, label='Original')
ax1.quiver(0,0,0, *y_3d, color='#3498db', lw=3, arrow_length_ratio=0.1, label='Rotated')
ax1.set_title('3D: rotation preserves length'); ax1.legend(fontsize=9)

ax2 = fig.add_subplot(132)
ax2.bar(range(30), x[:30], color='#e74c3c', alpha=0.8)
ax2.set_title(f'Original (first 30/{d} coords)'); ax2.set_ylim(-0.15, 0.15)

ax3 = fig.add_subplot(133)
ax3.bar(range(30), y[:30], color='#3498db', alpha=0.8)
ax3.set_title(f'After rotation (Gaussian, std≈{y.std():.3f})'); ax3.set_ylim(-0.15, 0.15)

plt.tight_layout(); plt.show()

## 2. Full Pipeline: Compress → Decompress
Multiple examples showing the 4-step round trip:

**x → Πx (rotate) → quantize → Πᵀ(quantized) ≈ x (inverse rotate)**

In [ ]:
# --- Math: Build quantizer + compress/decompress function ---
sigma = 1 / np.sqrt(d)
samp = np.random.randn(100000) * sigma

def build_codebook(b):
    centroids, _ = kmeans(samp, 2**b)
    return np.sort(centroids)

def compress(x, Pi, centroids):
    """x → rotate → snap each coord to nearest centroid → store indices"""
    y = Pi @ x
    indices = np.argmin(np.abs(y[:, None] - centroids[None, :]), axis=1)
    return indices, y  # indices are what gets stored (compressed)

def decompress(indices, Pi, centroids):
    """indices → look up centroids → inverse rotate"""
    y_hat = centroids[indices]
    return Pi.T @ y_hat, y_hat  # reconstructed vector + rotated version

codebooks = {b: build_codebook(b) for b in [1, 2, 3, 4]}
for b, cb in codebooks.items():
    print(f'{b}-bit codebook ({2**b} centroids): {cb.round(4)}')

In [ ]:
# --- Plot: Multiple examples at different bit-widths ---
# Generate 3 different random vectors
examples = []
for i in range(3):
    xi = np.random.randn(d); xi = xi / np.linalg.norm(xi)
    examples.append(xi)

fig, axes = plt.subplots(3, 5, figsize=(22, 10))
show = 25  # coords to show

for row, xi in enumerate(examples):
    # Original
    axes[row, 0].bar(range(show), xi[:show], color='#e74c3c', alpha=0.8)
    axes[row, 0].set_ylim(-0.15, 0.15)
    if row == 0: axes[row, 0].set_title('Original x')
    axes[row, 0].set_ylabel(f'Vector {row+1}', fontsize=12)
    
    for col, b in enumerate([1, 2, 3, 4]):
        idx, y_rot = compress(xi, Pi, codebooks[b])
        x_hat, y_hat = decompress(idx, Pi, codebooks[b])
        mse = np.mean((xi - x_hat)**2)
        
        ax = axes[row, col + 1]
        ax.bar(range(show), x_hat[:show], color='#2ecc71', alpha=0.7, label='reconstructed')
        ax.bar(range(show), xi[:show], color='#e74c3c', alpha=0.25, label='original')
        ax.set_ylim(-0.15, 0.15)
        if row == 0: ax.set_title(f'{b}-bit (MSE={mse:.5f})')
        else: ax.set_title(f'MSE={mse:.5f}', fontsize=10)
        if row == 0 and col == 0: ax.legend(fontsize=8)

plt.suptitle('Compress → Decompress at 1, 2, 3, 4 bits (3 different vectors)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# --- Plot: Detailed 4-step pipeline for one vector at 2 bits ---
xi = examples[0]
b = 2
y_rot = Pi @ xi
idx, _ = compress(xi, Pi, codebooks[b])
x_hat, y_hat = decompress(idx, Pi, codebooks[b])

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
s = 20

axes[0].bar(range(s), xi[:s], color='#e74c3c')
axes[0].set_title('1. Original x'); axes[0].set_ylim(-0.15, 0.15)

axes[1].bar(range(s), y_rot[:s], color='#3498db')
axes[1].set_title('2. Rotated Πx (≈Gaussian)'); axes[1].set_ylim(-0.15, 0.15)

axes[2].bar(range(s), y_hat[:s], color='#9b59b6')
for c_val in codebooks[b]:
    axes[2].axhline(c_val, color='orange', ls='--', lw=1, alpha=0.6)
axes[2].set_title(f'3. Quantized ({2**b} levels)'); axes[2].set_ylim(-0.15, 0.15)

axes[3].bar(range(s), x_hat[:s], color='#2ecc71', alpha=0.7, label='reconstructed')
axes[3].bar(range(s), xi[:s], color='#e74c3c', alpha=0.25, label='original')
axes[3].set_title(f'4. Inverse rotate Πᵀ ≈ x\nMSE={np.mean((xi-x_hat)**2):.5f}')
axes[3].set_ylim(-0.15, 0.15); axes[3].legend(fontsize=9)

plt.suptitle('x → Πx → quantize → Πᵀ(quantized) ≈ x', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 3. Scalar Quantization — What Centroids Mean

After rotation, each coordinate is ~Gaussian. The quantizer partitions the range into 2^b **buckets**, each covering roughly equal probability mass. The **centroid** is the average value within each bucket.

Centroids cluster near zero (where density is highest) — NOT evenly spaced. More bits = more buckets = finer resolution.

In [ ]:
# --- Math: Centroids and bucket probabilities ---
for b in [1, 2, 3, 4]:
    cb = codebooks[b]
    bounds = np.concatenate([[-np.inf], (cb[:-1]+cb[1:])/2, [np.inf]])
    probs = [norm.cdf(bounds[i+1], 0, sigma) - norm.cdf(bounds[i], 0, sigma) for i in range(2**b)]
    print(f'b={b}: centroids={cb.round(4)}, bucket probs={[f"{p:.2f}" for p in probs]}')

In [ ]:
# --- Plot: Buckets with colored fills ---
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
xx = np.linspace(-0.15, 0.15, 500)
pdf = norm.pdf(xx, 0, sigma)

for idx, b in enumerate([1, 2, 3, 4]):
    ax = axes[idx]
    cb = codebooks[b]
    bounds = np.concatenate([[-0.2], (cb[:-1]+cb[1:])/2, [0.2]])
    colors = plt.cm.Set2(np.linspace(0, 1, 2**b))
    for i in range(2**b):
        mask = (xx >= bounds[i]) & (xx < bounds[i+1])
        ax.fill_between(xx[mask], pdf[mask], alpha=0.4, color=colors[i])
        ax.axvline(cb[i], color=colors[i], lw=2.5)
    ax.plot(xx, pdf, 'k-', lw=1, alpha=0.4)
    ax.set_title(f'b={b} → {2**b} buckets')

plt.suptitle('Each color = bucket. Lines = centroids. More centroids near zero (highest density).', fontsize=13, y=1.04)
plt.tight_layout(); plt.show()

## 4. The Bias Problem
MSE-optimal quantizers give **biased** inner products (slope ≈ 2/π at 1 bit).  
Transformer attention computes Q·K — bias means wrong attention scores.

In [ ]:
# --- Math: Measure bias ---
n = 3000
rand_unit = lambda n, d: (v := np.random.randn(n, d)) / np.linalg.norm(v, axis=1, keepdims=True)
xs, ys = rand_unit(n, d), rand_unit(n, d)
true_ip = np.sum(xs * ys, axis=1)

xs_rot = (Pi @ xs.T).T
c = np.sqrt(2 / (np.pi * d))
xs_deq = (Pi.T @ (np.sign(xs_rot) * c).T).T
est_ip = np.sum(ys * xs_deq, axis=1)
err = est_ip - true_ip
lims = [-0.15, 0.15]

print(f'Bias slope: {np.polyfit(true_ip, est_ip, 1)[0]:.3f} (expected 2/π = {2/np.pi:.3f})')
print(f'Mean error: {err.mean():.4f} (should be 0 if unbiased)')

In [ ]:
# --- Plot: Bias ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(true_ip, est_ip, alpha=0.1, s=3, color='#e74c3c')
axes[0].plot(lims, lims, 'k--', lw=1.5, label='Perfect (y=x)')
axes[0].plot(lims, [l*2/np.pi for l in lims], 'b-', lw=2, label=f'Bias (slope≈{2/np.pi:.2f})')
axes[0].set(xlim=lims, ylim=lims, xlabel='True ⟨x,y⟩', ylabel='Estimated',
            title='Dots follow blue line, not black → biased')
axes[0].legend(); axes[0].set_aspect('equal')
axes[1].hist(err, bins=50, density=True, color='#e74c3c', alpha=0.7)
axes[1].axvline(0, color='k', ls='--'); axes[1].axvline(err.mean(), color='red', lw=2)
axes[1].set(xlabel='Error', title=f'Mean error = {err.mean():.4f} (should be 0)')
plt.tight_layout(); plt.show()

## 5. The Two-Stage Fix
**Stage 1**: (b-1) bits for MSE &nbsp;|&nbsp; **Stage 2**: 1 bit QJL on residual

MSE-only: low variance, **biased** (systematically wrong).  
Two-stage: higher variance, **unbiased** (correct on average).  
For attention, unbiasedness wins — bias consistently attends to wrong tokens.

In [ ]:
# --- Math: Two-stage ---
xs_mse = (Pi.T @ (np.sign(xs_rot) * c).T).T
est_mse_only = np.sum(ys * xs_mse, axis=1)

residuals = xs - xs_mse
r_norms = np.linalg.norm(residuals, axis=1)
S = np.random.randn(d, d)
qjl_signs = np.sign((S @ residuals.T).T)
xs_qjl = np.sqrt(np.pi/2) / d * (S.T @ qjl_signs.T).T * r_norms[:, None]
est_combined = np.sum(ys * (xs_mse + xs_qjl), axis=1)

err_mse = est_mse_only - true_ip
err_fix = est_combined - true_ip
print(f'MSE-only:  mean={err_mse.mean():+.4f} (biased),   std={err_mse.std():.4f}')
print(f'Two-stage: mean={err_fix.mean():+.4f} (unbiased), std={err_fix.std():.4f}')

In [ ]:
# --- Plot: Bias vs variance tradeoff (side by side + overlay) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(true_ip, est_mse_only, alpha=0.15, s=3, color='#e74c3c')
axes[0].plot(lims, lims, 'k--', lw=2)
axes[0].set(xlim=lims, ylim=lims, title=f'MSE-only: tight but OFFSET\nmean={err_mse.mean():+.4f}')
axes[0].set_aspect('equal')

axes[1].scatter(true_ip, est_combined, alpha=0.15, s=3, color='#2ecc71')
axes[1].plot(lims, lims, 'k--', lw=2)
axes[1].set(xlim=lims, ylim=lims, title=f'Two-stage: wider but CENTERED\nmean={err_fix.mean():+.4f}')
axes[1].set_aspect('equal')

axes[2].hist(err_mse, bins=50, density=True, alpha=0.5, color='#e74c3c', label='MSE-only (biased, tight)')
axes[2].hist(err_fix, bins=50, density=True, alpha=0.5, color='#2ecc71', label='Two-stage (unbiased, wider)')
axes[2].axvline(0, color='k', ls='--', lw=2)
axes[2].set(xlabel='Error', title='Bias vs Variance tradeoff'); axes[2].legend()
plt.tight_layout(); plt.show()

## 6. Near-Optimality
TurboQuant is within ~2.7× of the Shannon lower bound. No algorithm can beat the lower bound.

In [ ]:
# --- Math ---
bits = np.array([1, 2, 3, 4, 5])
lower = 1 / 4**bits
turbo = np.array([0.36, 0.117, 0.03, 0.009, 0.009/4])
for b, l, t in zip(bits, lower, turbo):
    print(f'b={b}: lower={l:.5f}, TurboQuant={t:.5f}, gap={t/l:.1f}×')

In [ ]:
# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(bits, lower, 's--', color='#2ecc71', lw=2, ms=10, label='Lower bound (impossible to beat)')
ax.semilogy(bits, turbo, 'o-', color='#3498db', lw=2, ms=10, label='TurboQuant')
ax.fill_between(bits, lower, turbo, alpha=0.15, color='#3498db')
for i, b in enumerate(bits[:4]):
    ax.annotate(f'{turbo[i]/lower[i]:.1f}×', xy=(b, np.sqrt(turbo[i]*lower[i])),
                fontsize=12, ha='center', color='#e74c3c', fontweight='bold')
ax.set(xlabel='Bit-width', ylabel='MSE Distortion (log)', title='Gap to theoretical limit')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. KV Cache: The Full GPU Memory Picture
Model weights are **fixed** (~16GB for Llama-8B). The KV cache grows **linearly** with context.  
At long contexts, the cache dominates GPU memory — that's where compression has the biggest impact.

In [ ]:
# --- Math: Memory breakdown ---
ctx = np.array([1, 4, 16, 32, 64, 100, 128]) * 1000
model_weights_gb = 8e9 * 2 / 1e9  # 8B params × 2 bytes (fp16) = 16 GB
kv_cache_gb = ctx * 32 * 8 * 128 * 2 * 2 / 1e9

print(f'Model weights (fixed): {model_weights_gb:.0f} GB')
for c, kv in zip(ctx, kv_cache_gb):
    total = model_weights_gb + kv
    print(f'  {c/1000:>4.0f}K ctx: weights={model_weights_gb:.0f}GB + cache={kv:.1f}GB = {total:.1f}GB total ({kv/total*100:.0f}% cache)')

In [ ]:
# --- Plot: Stacked memory + quality ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

ax = axes[0]
weights = np.full_like(kv_cache_gb, model_weights_gb)
ax.fill_between(ctx/1000, 0, weights, alpha=0.4, color='#95a5a6', label=f'Model weights ({model_weights_gb:.0f}GB fixed)')
ax.fill_between(ctx/1000, weights, weights + kv_cache_gb, alpha=0.4, color='#e74c3c', label='KV cache (16-bit)')
ax.plot(ctx/1000, weights + kv_cache_gb * 3.5/16, 's-', color='#3498db', lw=2.5, label='Total with TurboQuant 3.5-bit')
ax.axhline(80, color='black', ls=':', lw=2); ax.text(2, 83, 'A100 (80GB)', fontsize=10)
ax.axhline(24, color='black', ls=':', lw=1.5, alpha=0.6); ax.text(2, 26, 'RTX 4090 (24GB)', fontsize=10)
ax.set(xlabel='Context (K tokens)', ylabel='GPU Memory (GB)',
       title='Weights + KV cache\n(cache dominates at long contexts)')
ax.legend(loc='upper left', fontsize=10); ax.set_ylim(0, 180)

ax = axes[1]
methods = ['Full\n(16b)', 'KIVI\n(3b)', 'Polar\n(3.9b)', 'Turbo\n(2.5b)', 'Turbo\n(3.5b)']
scores = [50.06, 48.50, 49.78, 49.44, 50.06]
clrs = ['#95a5a6', '#e74c3c', '#f39c12', '#3498db', '#3498db']
ax.bar(methods, scores, color=clrs, alpha=0.8, edgecolor='black')
ax.set(ylabel='LongBench-E Avg Score', title='Quality vs compression')
ax.set_ylim(47, 51); ax.axhline(50.06, color='gray', ls='--', alpha=0.5)
ax.annotate('identical!', xy=(4, 50.15), fontsize=11, color='#3498db', fontweight='bold', ha='center')
plt.tight_layout(); plt.show()

## Buddy Pairs — Paper Reading Session

| Pair | Section |
|---|---|
| **Niloy + Maxwell** | Sec 3.3: Lower Bounds (Shannon + Yao minimax) |
| **Sid + Carter** | Sec 3.1-3.2: Algorithms 1 & 2 (MSE + inner product fix) |
| **Sebastian + Maksym** | Sec 4.2-4.3: KV cache experiments (Needle-in-Haystack + LongBench) |
| **Max Koster + Nathaniel** | Sec 4.4: Nearest neighbor search + indexing time |
| **Lucas + Saahil + Michelle** | Sec 1: Intro + motivation + "why does this matter" |

**Floating:** Jinyoung + Michael Nath

**Presentation order:**
1. Lucas + Saahil + Michelle — "Why does vector quantization matter?"
2. Niloy + Maxwell — "What's the theoretical floor?"
3. Sid + Carter — "How do the two algorithms work?"
4. Sebastian + Maksym — "Does it work for LLMs?"
5. Max Koster + Nathaniel — "Does it work for search?"